# Proof of Concept SIMENU — Perencana Menu Mingguan Hemat

**Mata kuliah:** Kecerdasan Buatan (F5A5) · **UTS/ETS Ganjil 2026/2027** · Soal 4c (Bab 5)

Notebook ini memverifikasi *inti keputusan* agen SIMENU pada sampel kasus kecil:

1. **Retrieval resep (RAG-lite)** dengan TF-IDF (`scikit-learn`) dan filter alergen.
2. **Pemetaan nama bahan** (string matching) ke nama baku dengan TF-IDF *character n-gram* (analog pencocokan semantik pada Marin dkk., 2025).
3. **Optimasi menu** sebagai *Integer Linear Programming* (`scipy.optimize.milp`, solver HiGHS) dengan batasan anggaran, alergen, waktu masak, energi, protein, dan variasi protein utama.
4. **Verifier** deterministik yang memeriksa ulang setiap rencana.
5. **Evaluasi mini** pada 30 profil rumah tangga sintetis.

> **Catatan data.** Seluruh harga, waktu masak, dan nilai gizi pada notebook ini adalah **data sampel ilustrasi** yang disusun penulis untuk menguji alur algoritma. Data tersebut **bukan** data pasar maupun data TKPI. Pada implementasi, harga berasal dari tool harga bertimestamp dan gizi dari basis komposisi pangan Indonesia.

In [1]:
import numpy as np, pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy, sklearn
print("numpy", np.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)

numpy 2.4.4 | scipy 1.17.1 | scikit-learn 1.8.0


**Cara menjalankan:** `pip install numpy scipy scikit-learn pandas` lalu *Run All* (Python 3.10+).

## 1. Data sampel resep (ilustrasi)

Satu baris = satu resep untuk 2 porsi. `cost` dalam rupiah, `pref` adalah skor preferensi 1–5 dari profil pengguna, `protein_utama` dipakai untuk batasan variasi.

In [2]:
recipes = [
 # id, nama, protein_utama, cost, menit, kcal/porsi, protein g/porsi, pref, alergen, bahan
 ("R01","Ayam kecap + tumis buncis","ayam",38500,35,520,32,5,[],"ayam kecap manis bawang putih buncis"),
 ("R02","Tempe orek + sayur bayam","tempe",21000,25,480,22,5,[],"tempe kecap cabai bayam bawang merah"),
 ("R03","Soto ayam bening","ayam",36000,40,410,34,5,[],"ayam kunyit serai kentang wortel kol bihun"),
 ("R04","Ikan nila bakar + lalapan","ikan",42000,35,500,38,5,["ikan"],"ikan nila kecap jeruk nipis timun selada"),
 ("R05","Telur balado + capcay","telur",28500,30,530,24,5,["telur"],"telur cabai tomat sayur capcay wortel kol"),
 ("R06","Nasi goreng sayur + telur","telur",24000,20,560,20,5,["telur"],"nasi telur bawang putih wortel buncis kecap"),
 ("R07","Sayur lodeh + tahu","tahu",33000,40,470,22,5,[],"tahu santan labu siam terong kacang panjang"),
 ("R08","Pepes tahu","tahu",25000,30,430,21,4,[],"tahu daun kemangi tomat bawang daun pisang"),
 ("R09","Tahu bacem + tumis kangkung","tahu",22000,40,450,20,4,[],"tahu kecap gula merah kangkung bawang putih"),
 ("R10","Nasi goreng sayur-tempe","tempe",20000,20,540,20,4,[],"nasi tempe bawang putih wortel buncis kecap"),
 ("R11","Tumis udang buncis","udang",30000,25,400,26,5,["udang"],"udang buncis bawang putih saus tiram"),
 ("R12","Sambal goreng udang","udang",45000,30,450,30,4,["udang"],"udang santan cabai tomat bawang"),
 ("R13","Rendang daging sapi","sapi",68000,120,620,36,5,[],"daging sapi santan cabai serai lengkuas"),
 ("R14","Gado-gado","tahu",26000,25,480,18,3,["kacang"],"tahu tempe kangkung tauge kentang bumbu kacang"),
 ("R15","Pecel lele","ikan",35000,35,520,30,3,["ikan","kacang"],"lele goreng sambal pecel kacang kol timun"),
 ("R16","Ayam goreng lengkuas","ayam",41000,45,560,33,4,[],"ayam lengkuas bawang putih ketumbar"),
 ("R17","Oseng tempe kemangi","tempe",19000,20,430,19,3,[],"tempe kemangi cabai bawang merah tomat"),
]
df = pd.DataFrame(recipes, columns=["id","nama","protein_utama","cost","menit","kcal","protein","pref","alergen","bahan"])
df[["id","nama","protein_utama","cost","menit","kcal","protein","pref","alergen"]]

     id                         nama protein_utama   cost  menit  kcal  protein  pref         alergen
0   R01    Ayam kecap + tumis buncis          ayam  38500     35   520       32     5              []
1   R02     Tempe orek + sayur bayam         tempe  21000     25   480       22     5              []
2   R03             Soto ayam bening          ayam  36000     40   410       34     5              []
3   R04    Ikan nila bakar + lalapan          ikan  42000     35   500       38     5          [ikan]
4   R05        Telur balado + capcay         telur  28500     30   530       24     5         [telur]
5   R06    Nasi goreng sayur + telur         telur  24000     20   560       20     5         [telur]
6   R07           Sayur lodeh + tahu          tahu  33000     40   470       22     5              []
7   R08                   Pepes tahu          tahu  25000     30   430       21     4              []
8   R09  Tahu bacem + tumis kangkung          tahu  22000     40   450       20   

## 2. Retrieval resep (RAG-lite) dan filter alergen

Query pengguna diubah menjadi vektor TF-IDF, dibandingkan dengan teks resep memakai *cosine similarity*, lalu resep beralergen dibuang **sebelum** masuk ke solver (hard constraint).

In [3]:
def retrieve(query, alergen=(), k=5):
    corpus = (df["nama"] + " " + df["bahan"]).tolist()
    vec = TfidfVectorizer(ngram_range=(1,2))
    X = vec.fit_transform(corpus)
    q = vec.transform([query])
    sim = cosine_similarity(q, X).ravel()
    out = df.assign(skor=sim.round(3))
    mask = out["alergen"].apply(lambda a: not (set(a) & set(alergen)))
    return out[mask].sort_values("skor", ascending=False).head(k)[["id","nama","skor","alergen"]]

print("Query: 'makan malam tahu tempe hemat' | alergi: udang, telur")
print(retrieve("makan malam tahu tempe hemat", alergen=("udang","telur")).to_string(index=False))

Query: 'makan malam tahu tempe hemat' | alergi: udang, telur
 id                    nama  skor  alergen
R14               Gado-gado 0.343 [kacang]
R10 Nasi goreng sayur-tempe 0.179       []
R08              Pepes tahu 0.174       []
R07      Sayur lodeh + tahu 0.172       []
R17     Oseng tempe kemangi 0.171       []


## 3. Pemetaan nama bahan (string matching)

Nama bahan dari input pengguna dipetakan ke nama baku. Skor di bawah ambang 0,45 ditandai **perlu konfirmasi manual** (tahap dua seperti pada Marin dkk., 2025).

In [4]:
baku = ["bawang putih","bawang merah","tempe","tahu","ayam","ikan nila","telur","kecap manis","cabai merah","santan","buncis","kangkung"]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4)).fit(baku)
B = vec.transform(baku)
def petakan(teks, ambang=0.45):
    s = cosine_similarity(vec.transform([teks]), B).ravel()
    i = int(s.argmax())
    return baku[i], round(float(s[i]),3), ("OK" if s[i] >= ambang else "KONFIRMASI MANUAL")
rows = [(t,)+petakan(t) for t in ["bwg putih","tempe goreng","ayam potong","cabe merah","ikan nilla","kecap","jahe"]]
pd.DataFrame(rows, columns=["input_pengguna","nama_baku","kemiripan","status"])

  input_pengguna     nama_baku  kemiripan             status
0      bwg putih  bawang putih      0.771                 OK
1   tempe goreng         tempe      0.944                 OK
2    ayam potong          ayam      0.903                 OK
3     cabe merah   cabai merah      0.786                 OK
4     ikan nilla     ikan nila      0.929                 OK
5          kecap   kecap manis      0.728                 OK
6           jahe         tempe      0.213  KONFIRMASI MANUAL

## 4. Model optimasi menu (ILP)

Variabel keputusan `x[r] ∈ {0,1}` (resep *r* dipilih). Untuk `D` hari makan malam:

* maksimalkan  Σ pref[r]·x[r]  (selisih biaya sebagai *tie-breaker*),
* Σ x[r] = D,
* Σ cost[r]·x[r] ≤ B  (anggaran),
* x[r] = 0 jika *r* beralergen terlarang atau waktu masak > T_max,
* K_min ≤ rata-rata kcal ≤ K_max dan rata-rata protein ≥ P_min,
* Σ x[r] ≤ ⌈D/3⌉ untuk setiap protein utama (variasi).

Bila tidak *feasible*, agen menghitung **anggaran minimum** dengan solver yang sama (objektif: minimalkan biaya).

In [5]:
import math
def solve(budget, alergen=(), hari=7, tmax=40, kmin=400, kmax=700, pmin=20, mode="pref"):
    n = len(df)
    cost = df["cost"].to_numpy(float); pref = df["pref"].to_numpy(float)
    if mode == "pref":   c = -pref*100 + cost/1000.0        # maksimalkan preferensi, tie-break biaya
    else:                c = cost                           # minimalkan biaya
    ub = np.ones(n)
    for i, r in df.iterrows():
        if set(r["alergen"]) & set(alergen) or r["menit"] > tmax: ub[i] = 0
    A, lo, hi = [], [], []
    A.append(np.ones(n));                       lo.append(hari);   hi.append(hari)
    A.append(cost);                             lo.append(-np.inf); hi.append(budget if mode=="pref" else np.inf)
    A.append(df["kcal"].to_numpy(float)-kmin);   lo.append(0);       hi.append(np.inf)
    A.append(df["kcal"].to_numpy(float)-kmax);   lo.append(-np.inf); hi.append(0)
    A.append(df["protein"].to_numpy(float)-pmin);lo.append(0);       hi.append(np.inf)
    cap = math.ceil(hari/3)
    for g in df["protein_utama"].unique():
        A.append((df["protein_utama"]==g).to_numpy(float)); lo.append(-np.inf); hi.append(cap)
    res = milp(c, constraints=LinearConstraint(np.array(A), lo, hi),
               integrality=np.ones(n), bounds=Bounds(np.zeros(n), ub))
    if not res.success: return None
    return df[np.round(res.x) == 1].copy()

def verifikasi(plan, budget, alergen=(), hari=7, tmax=40, kmin=400, kmax=700, pmin=20):
    if plan is None: return {"lolos": False, "alasan": "tidak ada rencana"}
    cek = {
      "jumlah_hari": len(plan) == hari,
      "anggaran":    plan["cost"].sum() <= budget,
      "alergen":     not any(set(a) & set(alergen) for a in plan["alergen"]),
      "waktu":       (plan["menit"] <= tmax).all(),
      "energi":      kmin <= plan["kcal"].mean() <= kmax,
      "protein":     plan["protein"].mean() >= pmin,
      "variasi":     (plan["protein_utama"].value_counts() <= math.ceil(hari/3)).all(),
    }
    cek = {k: bool(v) for k, v in cek.items()}
    return {"lolos": all(cek.values()), **cek}

def tampil(plan, judul):
    print(judul)
    print(plan[["id","nama","protein_utama","menit","cost"]].to_string(index=False))
    total = f"{plan['cost'].sum():,.0f}".replace(",", ".")
    print(f"TOTAL biaya: Rp{total} | rata-rata kcal: {plan['kcal'].mean():.0f} | rata-rata protein: {plan['protein'].mean():.1f} g")

### 4a. Skenario A — anggaran Rp250.000, 7 hari, pantangan udang

In [6]:
A = solve(250000, alergen=("udang",))
tampil(A, "Rencana skenario A")
print("Verifier:", verifikasi(A, 250000, alergen=("udang",)))

Rencana skenario A
 id                      nama protein_utama  menit  cost
R01 Ayam kecap + tumis buncis          ayam     35 38500
R02  Tempe orek + sayur bayam         tempe     25 21000
R03          Soto ayam bening          ayam     40 36000
R04 Ikan nila bakar + lalapan          ikan     35 42000
R05     Telur balado + capcay         telur     30 28500
R06 Nasi goreng sayur + telur         telur     20 24000
R07        Sayur lodeh + tahu          tahu     40 33000
TOTAL biaya: Rp223.000 | rata-rata kcal: 496 | rata-rata protein: 27.4 g
Verifier: {'lolos': True, 'jumlah_hari': True, 'anggaran': True, 'alergen': True, 'waktu': True, 'energi': True, 'protein': True, 'variasi': True}


### 4b. Skenario B — anggaran turun ke Rp200.000 dan muncul alergi telur (re-optimasi)

In [7]:
Bp = solve(200000, alergen=("udang","telur"))
tampil(Bp, "Rencana skenario B")
print("Verifier:", verifikasi(Bp, 200000, alergen=("udang","telur")))
ganti = sorted(set(A["nama"]) - set(Bp["nama"])); baru = sorted(set(Bp["nama"]) - set(A["nama"]))
print("\nMenu dikeluarkan:", ganti); print("Menu masuk     :", baru)

Rencana skenario B
 id                        nama protein_utama  menit  cost
R01   Ayam kecap + tumis buncis          ayam     35 38500
R02    Tempe orek + sayur bayam         tempe     25 21000
R03            Soto ayam bening          ayam     40 36000
R07          Sayur lodeh + tahu          tahu     40 33000
R08                  Pepes tahu          tahu     30 25000
R09 Tahu bacem + tumis kangkung          tahu     40 22000
R10     Nasi goreng sayur-tempe         tempe     20 20000
TOTAL biaya: Rp195.500 | rata-rata kcal: 471 | rata-rata protein: 24.4 g
Verifier: {'lolos': True, 'jumlah_hari': True, 'anggaran': True, 'alergen': True, 'waktu': True, 'energi': True, 'protein': True, 'variasi': True}

Menu dikeluarkan: ['Ikan nila bakar + lalapan', 'Nasi goreng sayur + telur', 'Telur balado + capcay']
Menu masuk     : ['Nasi goreng sayur-tempe', 'Pepes tahu', 'Tahu bacem + tumis kangkung']


### 4c. Skenario C — anggaran Rp120.000 (tidak mungkin dipenuhi)

In [8]:
C = solve(120000, alergen=("udang",))
print("Solver pada anggaran Rp120.000 ->", "INFEASIBLE" if C is None else "feasible")
Cmin = solve(10**9, alergen=("udang",), mode="biaya")
rp = f"{Cmin['cost'].sum():,.0f}".replace(",", ".")
print("Anggaran minimum yang mungkin (7 hari, pantangan udang): Rp" + rp)
print("Menu pada biaya minimum:", ", ".join(Cmin["nama"]))
print("Verifier biaya minimum:", verifikasi(Cmin, Cmin["cost"].sum(), alergen=("udang",)))

Solver pada anggaran Rp120.000 -> INFEASIBLE
Anggaran minimum yang mungkin (7 hari, pantangan udang): Rp157.000
Menu pada biaya minimum: Tempe orek + sayur bayam, Nasi goreng sayur + telur, Pepes tahu, Tahu bacem + tumis kangkung, Nasi goreng sayur-tempe, Gado-gado, Oseng tempe kemangi
Verifier biaya minimum: {'lolos': True, 'jumlah_hari': True, 'anggaran': True, 'alergen': True, 'waktu': True, 'energi': True, 'protein': True, 'variasi': True}


### 4d. Verifier menangkap rencana yang melanggar batasan

Contoh rencana 'tanpa solver' yang hanya mengambil 7 resep dengan preferensi tertinggi dan mengabaikan batasan.

In [9]:
naif = df.sort_values(["pref","cost"], ascending=[False, True]).head(7)
tampil(naif, "Rencana naif (preferensi tertinggi, tanpa batasan)")
print("Verifier:", verifikasi(naif, 250000, alergen=("udang",)))

Rencana naif (preferensi tertinggi, tanpa batasan)
 id                      nama protein_utama  menit  cost
R02  Tempe orek + sayur bayam         tempe     25 21000
R06 Nasi goreng sayur + telur         telur     20 24000
R05     Telur balado + capcay         telur     30 28500
R11        Tumis udang buncis         udang     25 30000
R07        Sayur lodeh + tahu          tahu     40 33000
R03          Soto ayam bening          ayam     40 36000
R01 Ayam kecap + tumis buncis          ayam     35 38500
TOTAL biaya: Rp211.000 | rata-rata kcal: 481 | rata-rata protein: 25.7 g
Verifier: {'lolos': False, 'jumlah_hari': True, 'anggaran': True, 'alergen': False, 'waktu': True, 'energi': True, 'protein': True, 'variasi': True}


## 5. Evaluasi mini pada 30 profil rumah tangga sintetis

Profil dibangkitkan acak (`seed=7`): anggaran Rp150.000–Rp300.000, 0–2 alergen dari {udang, telur, kacang, ikan}. Dibandingkan: **solver + verifier** vs **rencana naif** (7 resep berpreferensi tertinggi setelah filter alergen, tanpa memeriksa anggaran, waktu, dan variasi).

In [10]:
rng = np.random.default_rng(7)
semua = ["udang","telur","kacang","ikan"]
hasil = []
for _ in range(30):
    b = int(rng.integers(15, 31))*10000
    al = tuple(rng.choice(semua, size=int(rng.integers(0,3)), replace=False))
    p = solve(b, alergen=al)
    naive_plan = df[[not (set(a) & set(al)) for a in df["alergen"]]].sort_values(["pref","cost"], ascending=[False,True]).head(7)
    hasil.append(dict(anggaran=b, alergen=",".join(al) or "-",
                      feasible=p is not None,
                      lolos_verifier=bool(verifikasi(p,b,al)["lolos"]) if p is not None else None,
                      naif_lolos=bool(verifikasi(naive_plan,b,al)["lolos"])))
R = pd.DataFrame(hasil)
feas = R[R.feasible]
print(f"Profil feasible            : {len(feas)}/30")
print(f"Rencana solver lolos verifier : {int(feas.lolos_verifier.sum())}/{len(feas)}  (budget compliance & 0 pelanggaran alergen)")
print(f"Rencana naif lolos verifier   : {int(R.naif_lolos.sum())}/30")
print("Profil tidak feasible -> agen menyatakan batas anggaran minimum dan meminta persetujuan relaksasi:")
print(R[~R.feasible][["anggaran","alergen"]].to_string(index=False))

Profil feasible            : 28/30
Rencana solver lolos verifier : 28/28  (budget compliance & 0 pelanggaran alergen)
Rencana naif lolos verifier   : 17/30
Profil tidak feasible -> agen menyatakan batas anggaran minimum dan meminta persetujuan relaksasi:
 anggaran alergen
   150000       -
   150000   udang


## 6. Interpretasi

* **Skenario A** — solver memilih 7 menu yang tidak mengandung udang, seluruhnya ≤ 40 menit, total di bawah anggaran, dengan variasi protein utama terjaga.
* **Skenario B** — saat anggaran turun dan telur menjadi alergen, solver mengganti menu secara otomatis dan tetap memenuhi semua batasan; perubahan dapat dijelaskan kepada pengguna (menu keluar/masuk dan alasannya).
* **Skenario C** — agen **tidak mengarang rencana**; ia melaporkan anggaran minimum yang mungkin sehingga pengguna dapat memutuskan (sesuai aturan bisnis RB-03 dan penanganan situasi 'anggaran terlalu kecil').
* **Verifier** menandai rencana naif yang melanggar batasan, sehingga pemeriksaan deterministik terbukti diperlukan di luar LLM.
* **Keterbatasan PoC:** data kecil dan ilustrasi; belum ada LLM, vektor DB, harga nyata, pembulatan kemasan, stok dapur, maupun penjadwalan harian. Komponen tersebut dikerjakan pada tahap implementasi (Pertemuan 9–15).